# Demo Notebook — Risk Marker Analysis & Customer Propensity Scoring

This notebook runs the full analysis pipeline on **synthetic data** that mirrors the real schema.
All findings below reproduce results from the proprietary dataset.

**Run this notebook top-to-bottom** — it generates all figures saved to `outputs/figures/`.

> *Note: AUC values on synthetic data will appear higher than the ~0.92 reported in the README
> because synthetic data has a clean latent risk signal, whereas real data includes noise from
> imperfect QA normalisation and unmeasured confounders. All qualitative findings hold.*


## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency, spearmanr
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# ── Constants ──────────────────────────────────────────────────────────────────
MARKERS = ['Compliant', 'HBP_Hx', 'Hosp', 'HBP_Related',
           'KnowsBP', 'Meds', 'Followup', 'WaitRef']

MARKER_LABELS = {
    'Compliant':    'Compliant with treatment',
    'HBP_Hx':       'History of high BP',
    'Hosp':         'Hospitalised due to BP',
    'HBP_Related':  'Hypertension-related conditions',
    'KnowsBP':      'Knows blood pressure',
    'Meds':         'Taking BP medication',
    'Followup':     'Requires follow-up',
    'WaitRef':      'Waiting referral / investigation',
}

DECISION_ORDER  = ['standard', 'non-standard', 'refer/evidence_required', 'postpone', 'decline']
DECISION_COLORS = ['#4CAF50', '#FF9800', '#2196F3', '#9C27B0', '#F44336']

FIGURES = '../outputs/figures'

PALETTE = {
    'standard':                 '#4CAF50',
    'non-standard':             '#FF9800',
    'refer/evidence_required':  '#2196F3',
    'postpone':                 '#9C27B0',
    'decline':                  '#F44336',
}

def save(name):
    plt.tight_layout()
    plt.savefig(f'{FIGURES}/{name}', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {FIGURES}/{name}')


## Load & preprocess

Load the synthetic dataset (or substitute your own — the schema is documented in `data/README.md`).


In [ ]:
df = pd.read_csv('../data/synthetic_data.csv')

# BP columns: 'F' means no reading — convert to numeric + flag
for col, out in [('sys_pressure', 'sys_num'), ('dias_pressure', 'dias_num')]:
    df[out] = pd.to_numeric(df[col], errors='coerce')
    
df['bp_known'] = df['sys_num'].notna().astype(int)

# Median-impute missing BP (important: done AFTER flagging)
df['sys_num']  = df['sys_num'].fillna(df['sys_num'].median())
df['dias_num'] = df['dias_num'].fillna(df['dias_num'].median())

# Ordered categorical for decision
df['decision'] = pd.Categorical(df['decision'], categories=DECISION_ORDER, ordered=True)

print(f"Rows         : {len(df):,}")
print(f"Customers    : {df['customer_key'].nunique():,}")
print(f"Applications : {df['application_key'].nunique():,}")
print(f"Engines      : {df['enquiry_engine'].nunique()}")
print()
print("Decision distribution:")
dist = df['decision'].value_counts()
for d in DECISION_ORDER:
    c = dist.get(d, 0)
    print(f"  {d:<32} {c:>7,}  ({c/len(df)*100:.1f}%)")


## Plot 1 — Decision mix per engine

Each engine processes applications with its own question coverage and risk appetite.
This plot shows how the decision mix varies — a first indicator that engines are doing
different things with the same customer signals.


In [ ]:
engine_order = (df.groupby('enquiry_engine')['decision']
                  .apply(lambda x: (x == 'standard').mean())
                  .sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(11, 5))

bottom = np.zeros(len(engine_order))
for dec, col in zip(DECISION_ORDER, DECISION_COLORS):
    vals = [(df[df['enquiry_engine'] == e]['decision'] == dec).mean() * 100
            for e in engine_order]
    ax.bar(engine_order, vals, bottom=bottom, color=col, label=dec, width=0.65)
    bottom += np.array(vals)

ax.set_xlabel('Engine', fontsize=12)
ax.set_ylabel('Share of decisions (%)', fontsize=12)
ax.set_title("Decision mix by engine\n(sorted by % standard, high → low)", fontsize=13)
ax.legend(loc='upper right', fontsize=9, framealpha=0.9)
ax.set_ylim(0, 105)
ax.tick_params(axis='x', rotation=30)
sns.despine()
save('01_decision_mix_by_engine.png')


## Plot 2 — Feature association ranking (Cramér's V)

How strongly does each feature associate with the final decision?

**Cramér's V** for categorical features (markers + gender + smoker): 0 = no association, 1 = perfect.  
**η² (eta-squared)** for continuous features: proportion of variance in that feature explained by decision.

The key finding: **all 8 risk markers outrank every demographic** on association strength.
Behavioural / procedural attributes (compliance, referral status) dominate physiological ones (age, BMI).


In [ ]:
def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = ct.sum().sum()
    r, k = ct.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1)))

def eta_squared(x, y):
    groups = [x[y == g] for g in y.unique()]
    grand_mean = x.mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total   = ((x - grand_mean)**2).sum()
    return ss_between / ss_total if ss_total > 0 else 0

rows = []
for m in MARKERS:
    rows.append({'feature': MARKER_LABELS[m], 'v': cramers_v(df[m], df['decision']), 'type': 'marker'})

for col, lbl in [('gender','Gender'), ('smoker','Smoker')]:
    rows.append({'feature': lbl, 'v': cramers_v(df[col], df['decision']), 'type': 'demo'})

for col, lbl in [('age_at_application','Age'), ('bmi','BMI'),
                  ('sys_num','Systolic BP'), ('dias_num','Diastolic BP')]:
    rows.append({'feature': lbl, 'v': eta_squared(df[col], df['decision']), 'type': 'demo'})

assoc = pd.DataFrame(rows).sort_values('v', ascending=True)

colors = ['#1565C0' if t == 'marker' else '#78909C' for t in assoc['type']]

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(assoc['feature'], assoc['v'], color=colors, edgecolor='white', height=0.7)

# Annotate values
for bar, val in zip(bars, assoc['v']):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='#1565C0', label='Risk marker'),
                   Patch(color='#78909C', label='Demographic')],
          fontsize=10, loc='lower right')

ax.set_xlabel("Cramér's V  /  η²  (association with decision)", fontsize=11)
ax.set_title("Feature association with final decision\nAll 8 markers outrank every demographic", fontsize=12)
ax.set_xlim(0, assoc['v'].max() * 1.2)
sns.despine()
save('02_feature_association_cramers_v.png')


## Plot 3 — Profile heatmap: % True per marker × decision

For each (marker, decision) combination: what share of customers in that decision class have the marker set to True?

Reveals the **signature pattern** of each outcome — declines cluster on WaitRef and non-compliance;
standards are uniformly low across all markers.


In [ ]:
pct_T = {}
for m in MARKERS:
    pct_T[MARKER_LABELS[m]] = {
        dec: (df.loc[df['decision'] == dec, m] == 'T').mean() * 100
        for dec in DECISION_ORDER
    }

hm_df = pd.DataFrame(pct_T).T.reindex(columns=DECISION_ORDER)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(hm_df, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': '% customers with marker = True'})
ax.set_xlabel('Decision', fontsize=11)
ax.set_ylabel('')
ax.set_title('% True per marker × decision class\n(higher = marker more prevalent in that outcome)', fontsize=12)
ax.tick_params(axis='x', rotation=20)
save('03_profile_heatmap.png')


## Plot 4 — The identity funnel

How much customer-side context is needed before decisions become deterministic?

We build progressively richer identity keys and count what share of
`(identity × engine)` combinations always produce one decision.


In [ ]:
df['profile'] = df[MARKERS].apply(lambda r: '|'.join(r.astype(str)), axis=1)

layers = [
    ('Markers only\n(no engine)',         ['profile']),
    ('+ Demographics\n(age, BMI, BP)',    ['profile', 'age_at_application', 'bmi', 'sys_num', 'dias_num']),
    ('+ Engine',                           ['profile', 'age_at_application', 'bmi', 'sys_num', 'dias_num',
                                            'enquiry_engine']),
]

results = []
for label, key_cols in layers:
    purity = (df.groupby(key_cols, observed=True)['decision']
                .nunique().reset_index())
    n_pure = (purity['decision'] == 1).sum()
    n_rows_pure = purity.loc[purity['decision'] == 1].merge(
        df.groupby(key_cols, observed=True).size().reset_index(name='n'), on=key_cols
    )['n'].sum() if len(key_cols) > 1 else (
        df[df['profile'].isin(purity.loc[purity['decision'] == 1, 'profile'])].shape[0]
    )
    results.append({'layer': label, 'pct': n_rows_pure / len(df) * 100})

labels  = [r['layer'] for r in results]
values  = [r['pct']   for r in results]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, values, color=['#1565C0', '#1976D2', '#42A5F5'], width=0.5, edgecolor='white')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.0f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

ax.set_ylabel('% of rows where outcome is deterministic', fontsize=11)
ax.set_title('Identity funnel\nHow much context is needed to predict the decision?', fontsize=12)
ax.set_ylim(0, 105)
ax.axhline(100, color='gray', linestyle='--', linewidth=0.8)
sns.despine()
save('04_identity_funnel.png')


## Plot 5 — Customer propensity score distribution

Train a random forest on **standards vs declines only** (anchor populations) to produce
`p_decline_like`: a 0–1 score for each customer.

Then score the full dataset and plot the distribution by decision class.


In [ ]:
# --- Feature matrix ---
anchor = df[df['decision'].isin(['standard', 'decline'])].copy()

feat_cols = []
for m in MARKERS:
    for v in ['T', 'F', 'NAsk']:
        col = f'{m}_{v}'
        anchor[col] = (anchor[m] == v).astype(int)
        df[col]     = (df[m] == v).astype(int)
        feat_cols.append(col)

feat_cols += ['age_at_application', 'bmi', 'sys_num', 'dias_num', 'bp_known']
anchor['gender_M'] = (anchor['gender'] == 'M').astype(int)
anchor['smoker_Y'] = (anchor['smoker'] == 'Y').astype(int)
df['gender_M']     = (df['gender'] == 'M').astype(int)
df['smoker_Y']     = (df['smoker'] == 'Y').astype(int)
feat_cols += ['gender_M', 'smoker_Y']

X_anchor = anchor[feat_cols].values
y_anchor = (anchor['decision'] == 'decline').astype(int).values
groups   = anchor['application_key'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X_anchor, y_anchor, groups))

rf = RandomForestClassifier(n_estimators=200, max_depth=8, min_samples_leaf=50,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_anchor[tr_idx], y_anchor[tr_idx])
auc = roc_auc_score(y_anchor[te_idx], rf.predict_proba(X_anchor[te_idx])[:, 1])
print(f"AUC (standards vs declines, no engine): {auc:.3f}")

df['p_decline_like'] = rf.predict_proba(df[feat_cols].values)[:, 1]

# --- Plot ---
fig, ax = plt.subplots(figsize=(11, 5))
bins = np.linspace(0, 1, 41)

for dec in DECISION_ORDER:
    sub = df.loc[df['decision'] == dec, 'p_decline_like']
    ax.hist(sub, bins=bins, alpha=0.65, color=PALETTE[dec],
            label=f'{dec}  (n={len(sub):,})', density=True, edgecolor='none')

ax.set_xlabel('p_decline_like  (0 = looks like standard, 1 = looks like decline)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Propensity score distribution by decision class\n'
             'Note: score is a relative position, not a real-world decline probability', fontsize=12)
ax.legend(fontsize=9, framealpha=0.9)
sns.despine()
save('05_score_distribution_by_decision.png')


## Plot 6 — Score distribution by engine (portability check)

If the score is truly customer-side (not engine-specific), the score distributions
should look similar across all 10 engines. This is the portability sanity check.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
eng_order = df.groupby('enquiry_engine')['p_decline_like'].median().sort_values().index

sns.boxplot(data=df, x='enquiry_engine', y='p_decline_like',
            order=eng_order, ax=ax, showfliers=False,
            palette=['#1565C0'] * len(eng_order))

ax.set_xlabel('Engine', fontsize=11)
ax.set_ylabel('p_decline_like', fontsize=11)
ax.set_title('Propensity score by engine\nSimilar distributions = score is portable across engines', fontsize=12)
ax.tick_params(axis='x', rotation=30)
sns.despine()
save('06_score_by_engine.png')


## Plot 7 — Loading severity vs propensity score

Does the mortality loading (em_load) track marker-based risk within non-standard decisions?

**Finding**: Spearman ρ ≈ 0.05 pooled across engines — loading does **not** reflect
the same risk gradient that the markers capture. The markers identify *whether* a
customer is high-risk; they do not grade severity within the non-standard band.
Loading is engine-specific pricing convention, not a portable risk scale.


In [ ]:
ns = df[(df['decision'] == 'non-standard') & df['em_load'].notna()].copy()
ns['em_load_num'] = pd.to_numeric(ns['em_load'], errors='coerce')
ns = ns.dropna(subset=['em_load_num'])

rho_pooled, _ = spearmanr(ns['em_load_num'], ns['p_decline_like'])
print(f"Spearman ρ (em_load vs p_decline_like), pooled: {rho_pooled:.3f}")

print("\nPer-engine ρ:")
for eng in sorted(ns['enquiry_engine'].unique()):
    sub = ns[ns['enquiry_engine'] == eng]
    if len(sub) > 30:
        r, _ = spearmanr(sub['em_load_num'], sub['p_decline_like'])
        print(f"  {eng}: ρ = {r:.3f}  (n={len(sub):,})")

# Band the loading for the plot
ns['em_band'] = pd.cut(ns['em_load_num'],
                        bins=[-np.inf, 35, 60, 101, np.inf],
                        labels=['Light (≤35)', 'Moderate (≤60)', 'Heavy (≤101)', 'Very Heavy (>101)'])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: scatter (sample for clarity)
sample = ns.sample(min(5000, len(ns)), random_state=42)
axes[0].scatter(sample['em_load_num'], sample['p_decline_like'],
                alpha=0.15, s=8, color='#1565C0')
axes[0].set_xlabel('em_load (mortality loading %)', fontsize=11)
axes[0].set_ylabel('p_decline_like', fontsize=11)
axes[0].set_title(f'Loading vs propensity score\nSpearman ρ = {rho_pooled:.3f} (pooled)', fontsize=11)
m, b = np.polyfit(sample['em_load_num'], sample['p_decline_like'], 1)
x_ = np.linspace(sample['em_load_num'].min(), sample['em_load_num'].max(), 100)
axes[0].plot(x_, m*x_ + b, color='#F44336', linewidth=1.5, label='trend')
axes[0].legend(fontsize=9)

# Right: boxplot by band
band_order = ['Light (≤35)', 'Moderate (≤60)', 'Heavy (≤101)', 'Very Heavy (>101)']
sns.boxplot(data=ns, x='em_band', y='p_decline_like', order=band_order,
            ax=axes[1], showfliers=False,
            palette=['#AED6F1', '#5DADE2', '#2E86C1', '#1A5276'])
axes[1].set_xlabel('Loading band', fontsize=11)
axes[1].set_ylabel('p_decline_like', fontsize=11)
axes[1].set_title('Score by loading band (non-standard)\nFlat = markers do not grade NS severity', fontsize=11)
axes[1].tick_params(axis='x', rotation=15)

for ax in axes:
    sns.despine(ax=ax)

save('07_emload_vs_score.png')


## Summary of findings

| Finding | Value |
|---|---|
| % decisions deterministic (markers only) | ~45–65% |
| % decisions deterministic (markers + demographics) | ~80–85% |
| % decisions deterministic (markers + demographics + engine) | ~95%+ |
| Top marker association (Cramér's V) | WaitRef ≈ 0.29 |
| Best demographic association | Smoker ≈ 0.06 |
| AUC — standards vs declines (no engine) | ~0.99 (synthetic) / ~0.92 (real) |
| Spearman ρ: em_load vs p_decline_like (pooled) | ~0.05 |
| % impure (profile × engine) cells | ~43% |

All figures saved to `outputs/figures/`.
